# 01 — Load the eight samples

Equivalent to `scripts/01_load_data.py`. Merges the 8 CellRanger outputs into one
AnnData and attaches sex / treatment / replicate to every cell.

## Two data problems handled here

Both were found by debugging, not by reading documentation, and both fail silently
if left alone.

**1. The folder names do not match their contents.** Sex markers separate the eight
samples by the *treatment* field of the folder name, not the *sex* field. Corrected
from the deposition order in the authors' published R code. See `config.SAMPLE_REMAP`
and `docs/data_provenance.md`.

**2. A gene named `nan`.** Nanchung (`Dmel_CG5842`) has the symbol `nan`, which
pandas reads as a missing value. Left alone it breaks the h5ad write and would
silently fail every marker lookup downstream.

A third problem — space-delimited `features.tsv.gz` — is fixed once, upstream, by
`tools/fix_features_separator.py`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import config

sc.settings.verbosity = 3
sc.settings.figdir = config.FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.logging.print_header()

## Find the sample folders

In [ ]:
sample_dirs = sorted(
    p for p in config.DATA_DIR.iterdir()
    if p.is_dir() and p.name != 'original' and (p / 'matrix.mtx.gz').exists()
)
print(f'Found {len(sample_dirs)} samples:')
for p in sample_dirs:
    print(' ', p.name)

## The label correction

Read this table before running the loader. The left column is what the folder says;
the right is what the data actually contains.

In [ ]:
remap = pd.DataFrame(
    [(k, *v) for k, v in config.SAMPLE_REMAP.items()],
    columns=['folder', 'TRUE_sex', 'TRUE_treatment', 'replicate', 'geo_position'],
)
remap['label_said_sex'] = [f.split('_')[0] for f in remap['folder']]
remap['label_said_treatment'] = [f.split('_')[1] for f in remap['folder']]
remap[['folder', 'label_said_sex', 'label_said_treatment',
       'TRUE_sex', 'TRUE_treatment', 'geo_position']]

## Load each sample

Note what this does **not** do: it never parses the folder name for sex or
treatment. Doing that is precisely what produced the wrong answer. The remap table
is the authority, and an unrecognised folder stops the run rather than being
guessed at.

In [ ]:
adatas = {}

for path in sample_dirs:
    folder = path.name
    if folder not in config.SAMPLE_REMAP:
        raise KeyError(f'{folder} not in SAMPLE_REMAP — verify it, do not guess')
    sex, treatment, replicate, geo = config.SAMPLE_REMAP[folder]
    sample_id = f'{sex}_{treatment}_R{replicate}'

    a = sc.read_10x_mtx(path, var_names='gene_symbols', cache=False)
    a.var_names_make_unique()

    a.obs['sample'] = sample_id
    a.obs['sample_folder'] = folder
    a.obs['sex'] = sex
    a.obs['treatment'] = treatment
    a.obs['replicate'] = replicate
    a.obs['geo_position'] = geo
    a.obs['condition'] = f'{sex}_{treatment}'

    print(f'{folder:>18} -> {sample_id:<22} ({geo})  {a.n_obs:,} cells')
    adatas[sample_id] = a

## Concatenate

`join='outer'` keeps every gene seen in any sample. `join='inner'` would silently
drop genes missing from a single sample — possibly a marker you need later.
`index_unique='-'` stops identical barcodes from different runs colliding.

In [ ]:
adata = ad.concat(adatas, label='sample_batch', index_unique='-', join='outer')
adata.var_names_make_unique()
print(f'Merged: {adata.n_obs:,} cells x {adata.n_vars:,} genes')

## Recover the gene lost to NA parsing

`adata.var_names.isna()` should find exactly one — nanchung. If it finds more,
stop and look at which genes they are rather than patching blindly.

In [ ]:
bad = pd.isna(adata.var_names)
print('unnamed genes:', int(bad.sum()))

if bad.any():
    names = pd.Series(adata.var_names, dtype=object)
    names[bad] = list(config.GENE_NAME_NA_FIXES.values())[:int(bad.sum())]
    adata.var_names = pd.Index(names.astype(str))
    adata.var_names_make_unique()
    print('recovered as:', list(config.GENE_NAME_NA_FIXES.values()))

assert not pd.isna(adata.var_names).any(), 'unnamed gene survived — stop'
print("'na' (narrow abdomen) still present:", 'na' in adata.var_names)

In [ ]:
for col in ['sample', 'sample_folder', 'sex', 'treatment',
            'replicate', 'geo_position', 'condition']:
    adata.obs[col] = adata.obs[col].astype('category')

adata

## Check the design survived

This must be a full 2×2×2 — eight non-zero entries. A zero anywhere means a
sample failed to load.

In [ ]:
pd.crosstab(adata.obs['sex'], [adata.obs['treatment'], adata.obs['replicate']])

In [ ]:
adata.obs['sample'].value_counts().sort_index()

In [ ]:
adata.write(config.H5AD_RAW)
print('Wrote', config.H5AD_RAW)

---
**Next:** run `python tools/verify_remap.py` in a terminal. It confirms the
corrected labels against sex markers and against the paper's independent finding
that males respond more than females. Then `02_qc_filter.ipynb`.